In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql import functions as F
import os


In [ ]:
try:
    spark.stop()
except:
    pass


In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_Condition")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.config("spark.sql.session.timeZone", "UTC")
.master("local[*]")
.getOrCreate())


In [ ]:
silver_base_path = "../../data_lake/silver/silver_condition/"
gold_base_path = "../../data_lake/gold/dim_condition/"
gold_dimpatient = "../../data_lake/gold/dim_patient/"
gold_dimdate = "../../data_lake/gold/dim_date/"


In [ ]:
df_condition = spark.read.format("parquet").load(silver_base_path)
df_dimpatient = spark.read.format("parquet").load(gold_dimpatient)
df_dimdate = spark.read.format("parquet").load(gold_dimdate)


In [ ]:
df_inter = (df_condition.alias("cond")
    .join(df_dimpatient.alias("pat"), col("cond.patient_id") == col("pat.patient_id"), "left")
    .join(df_dimdate.alias("d_onset"), col("cond.onset_date_time").cast("date") == col("d_onset.date"), "left")
    .join(df_dimdate.alias("d_recorded"), col("cond.recorded_date").cast("date") == col("d_recorded.date"), "left")
    .join(df_dimdate.alias("d_abatement"), col("cond.abatement_date_time").cast("date") == col("d_abatement.date"), "left")
    .select(
        F.conv(F.substring(F.md5(col("cond.condition_id")), 1, 15), 16, 10).cast("bigint").alias("condition_key"),
        col("cond.condition_id"),
        col("pat.patient_key"),
        F.conv(F.substring(F.md5(col("cond.encounter_id")), 1, 15), 16, 10).cast("bigint").alias("encounter_key"),
        col("cond.clinical_status_coding"),
        col("cond.verification_status_coding"),
        col("cond.category_coding"),
        col("cond.code_coding"),
        col("d_onset.date_key").alias("onset_date_key"),
        col("d_recorded.date_key").alias("recorded_date_key"),
        col("d_abatement.date_key").alias("abatement_date_key"),
        F.current_timestamp().alias("gold_timestamp")
    )
)


In [ ]:
df_inter.write.mode("overwrite").format("parquet").save(gold_base_path)


In [ ]:
spark.stop()
